# 16 · Corpus80 npz parça 0: 30 küçük tar arşivi (bağlantı kopmalarına dayanıklı indirme) (Kaggle, CPU)

**Neden?** Kaggle CLI kernel çıktısındaki her dosyayı ayrı istekle indiriyor; 1.469 küçük `.npz`
dosyası dakikada ~20 dosya hızla ~1 saat sürüyor. Aynı veriyi 3 büyük tar arşivinde toplamak indirmeyi
bant genişliğiyle sınırlı hâle getirir (birkaç dakika). Arşivler sıkıştırılmaz (npz zaten sıkıştırılmış).

In [ ]:
import os, tarfile, time
from pathlib import Path
import numpy as np

SHARD, N_PARTS = 0, 30
def find_dir(name):
    for d, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
        if Path(d).name == name and any(f.endswith(".npz") for f in files):
            return Path(d)
    raise FileNotFoundError(name)
src = find_dir("npz")
files = sorted(src.glob("*.npz"))
print(src, len(files))
t0 = time.time()
for p, chunk in enumerate(np.array_split(np.array(files), N_PARTS)):
    out = Path(f"/kaggle/working/c80_s{SHARD}_p{p:02d}.tar")
    with tarfile.open(out, "w") as tar:
        for f in chunk:
            tar.add(f, arcname=Path(f).name)
    print(out.name, len(chunk), f"{out.stat().st_size / 1e9:.2f} GB", f"{time.time() - t0:.0f}s", flush=True)
for name in [f"masks_s{SHARD:02d}.npy", f"ids_s{SHARD:02d}.npy"]:
    hit = next(Path("/kaggle/input").rglob(name))
    Path("/kaggle/working", name).write_bytes(hit.read_bytes())
print(sorted(os.listdir("/kaggle/working")))